# Binary refrigeration fault classifier

Run from the project root with Python 3, pandas, NumPy, LightGBM and scikit-learn. Run all cells. The six air-temperature inputs are listed below. The dataset has one recording per fault, so within-run test blocks measure same-recording performance. The default uses **weak run-level labels**: all windows from fault files are treated as fault-scenario windows. Scores measure agreement with that proxy and cannot establish active-fault detection or time to alert. Supply verified intervals below when available.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score

AIR = [
    "T-LTcase-Ret", "T-LTcase-Sup", "T-MTCase-Ret", "T-MTCase-Sup",
    "T-GC-Fan2-In", "T-GC-Fan2-Out",
]
LOOKBACK = 1  # completed one-minute bins per feature vector
# Optional verified fault-active intervals keyed by filename stem.
# Use [(start, end), ...] with full timestamps and exclusive ends.
FAULT_INTERVALS = {}
ALLOW_WEAK_LABELS = True
BASELINE_VALIDATION = "BaselineTestD"
BASELINE_TEST = "BaselineTestE"

paths = sorted(Path("data").glob("*.csv"))
if sum(p.stem.startswith("Fault") for p in paths) != 6 or sum(
    p.stem.startswith("BaselineTest") for p in paths
) != 5:
    raise ValueError("Expected the six fault and five baseline CSVs under data/.")

## Construct one-minute windows

Aggregate all readings in each recorded minute, retaining file order for first-to-last change where timestamps repeat. Compute the median, population standard deviation and first-to-last change for the six channels and three air temperature differences. Require at least two readings per quantity. A prediction is available at the end of each minute. Keep a one-minute purge between fault split blocks.

In [2]:
def make_windows(path):
    raw = pd.read_csv(path, usecols=["Timestamp", *AIR])
    stamps = pd.to_datetime(raw.pop("Timestamp"), format="mixed", errors="raise")
    raw = raw.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    raw["Freezer air drop"] = raw[AIR[0]] - raw[AIR[1]]
    raw["Chilled air drop"] = raw[AIR[2]] - raw[AIR[3]]
    raw["Gas cooler air rise"] = raw[AIR[5]] - raw[AIR[4]]
    ordered = raw.iloc[np.argsort(stamps.to_numpy(), kind="stable")]
    bins = stamps.sort_values(kind="stable").dt.floor("min")
    grouped = ordered.groupby(bins)
    minute = grouped.median()
    features = pd.concat([
        minute.add_suffix(" median"),
        grouped.std(ddof=0).add_suffix(" std"),
        (grouped.last() - grouped.first()).add_suffix(" change"),
    ], axis=1)
    features = features.loc[grouped.count().ge(2).all(axis=1)].copy()
    features["End"] = features.index + pd.Timedelta(minutes=1)
    features["Start"] = features["End"] - pd.Timedelta(minutes=LOOKBACK)
    run = path.stem
    features["Run"] = run
    features["Anomaly"] = int(run.startswith("Fault"))
    if not run.startswith("Fault"):
        features["Split"] = {
            BASELINE_VALIDATION: "validation", BASELINE_TEST: "test",
        }.get(run, "train")
        features["Label source"] = "baseline recording"
        return features

    start, stop = minute.index[0], minute.index[-1] + pd.Timedelta(minutes=1)
    first = start + (stop - start) * 0.60
    second = start + (stop - start) * 0.80
    gap = pd.Timedelta(minutes=LOOKBACK)
    features["Split"] = np.select(
        [features["End"].le(first),
         features["Start"].ge(first + gap) & features["End"].le(second),
         features["Start"].ge(second + gap)],
        ["train", "validation", "test"], default="purge",
    )
    features = features.loc[features["Split"].ne("purge")].copy()
    if run in FAULT_INTERVALS:
        active = pd.Series(False, index=features.index)
        for begin, end in FAULT_INTERVALS[run]:
            active |= features["Start"].ge(pd.Timestamp(begin)) & features["End"].le(pd.Timestamp(end))
        features = features.loc[active].copy()
        features["Label source"] = "supplied fault interval"
    elif ALLOW_WEAK_LABELS:
        features["Label source"] = "fault recording proxy"
    else:
        return features.iloc[:0]
    return features

windows = pd.concat([make_windows(path) for path in paths], ignore_index=True)
metadata = ["Run", "Start", "End", "Split", "Anomaly", "Label source"]
feature_names = windows.columns.drop(metadata).tolist()
display(windows.groupby(["Run", "Label source", "Split"]).size().unstack(fill_value=0))

train, validation, test = [windows.loc[windows["Split"].eq(split)].copy()
                           for split in ("train", "validation", "test")]
for frame in (train, validation, test):
    if frame["Anomaly"].nunique() != 2:
        raise ValueError("Each split needs both classes. Review fault intervals and split boundaries.")
for earlier, later in ((train, validation), (validation, test)):
    bounds = earlier.groupby("Run")["End"].max().to_frame().join(
        later.groupby("Run")["Start"].min(), how="inner")
    assert (bounds["Start"] - bounds["End"] >= pd.Timedelta(minutes=LOOKBACK)).all()

,Split,test,train,validation
Run,Label source,,,
BaselineTestA,baseline recording,0,1440,0
BaselineTestB,baseline recording,0,1440,0
BaselineTestC,baseline recording,0,1440,0
BaselineTestD,baseline recording,0,0,1440
BaselineTestE,baseline recording,1440,0,0
Fault1_DisplayCaseDoorOpen,fault recording proxy,287,864,287
Fault2_IceAccumulation,fault recording proxy,287,864,287
Fault3_EvapValveFailure,fault recording proxy,287,864,287
Fault4_MTEvapFanFailure,fault recording proxy,287,864,287


In [3]:
model = LGBMClassifier(
    objective="binary", n_estimators=100, learning_rate=0.05,
    num_leaves=15, min_child_samples=50, class_weight="balanced",
    random_state=42, n_jobs=2, verbosity=-1,
)
model.fit(train[feature_names], train["Anomaly"])

validation_scores = model.predict_proba(validation[feature_names])[:, 1]
precision, recall, thresholds = precision_recall_curve(validation["Anomaly"], validation_scores)
f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
threshold = float(thresholds[np.argmax(f1)])
print(f"Validation-selected threshold (maximum window F1): {threshold:.4f}")

test["Score"] = model.predict_proba(test[feature_names])[:, 1]
test["Alert"] = test["Score"].ge(threshold)
tn, fp, fn, tp = confusion_matrix(test["Anomaly"], test["Alert"], labels=[0, 1]).ravel()
print(f"Test ROC AUC: {roc_auc_score(test['Anomaly'], test['Score']):.3f}")
print(f"Test average precision: {average_precision_score(test['Anomaly'], test['Score']):.3f}")
print(f"Test window counts: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

# An alert starts when the score crosses above threshold, including after a gap.
previous_alert = test.groupby("Run")["Alert"].shift(fill_value=False)
step = test.groupby("Run")["End"].diff()
test["Alert start"] = test["Alert"] & (~previous_alert | step.ne(pd.Timedelta(minutes=1)))
by_run = test.groupby(["Run", "Label source"]).agg(
    Windows=("Alert", "size"), Alert_fraction=("Alert", "mean"), Alert_starts=("Alert start", "sum")
)
by_run["Alert starts per scored hour"] = 60 * by_run["Alert_starts"] / by_run["Windows"]
display(by_run.round(3))
display(test[["Run", "End", "Label source", "Score", "Alert"]].head())

Validation-selected threshold (maximum window F1): 0.0251
Test ROC AUC: 1.000
Test average precision: 1.000
Test window counts: TN=1380, FP=60, FN=0, TP=1722


,,Windows,Alert_fraction,Alert_starts,Alert starts per scored hour
Run,Label source,,,,
BaselineTestE,baseline recording,1440,0.042,29,1.208
Fault1_DisplayCaseDoorOpen,fault recording proxy,287,1.000,1,0.209
Fault2_IceAccumulation,fault recording proxy,287,1.000,1,0.209
Fault3_EvapValveFailure,fault recording proxy,287,1.000,1,0.209
Fault4_MTEvapFanFailure,fault recording proxy,287,1.000,1,0.209
Fault5_CondAPBlock,fault recording proxy,287,1.000,1,0.209
Fault6_MTEvapAPBlock,fault recording proxy,287,1.000,1,0.209


,Run,End,Label source,Score,Alert
5760,BaselineTestE,2019-10-26 00:01:00,baseline recording,0.021867,False
5761,BaselineTestE,2019-10-26 00:02:00,baseline recording,0.022774,False
5762,BaselineTestE,2019-10-26 00:03:00,baseline recording,0.024087,False
5763,BaselineTestE,2019-10-26 00:04:00,baseline recording,0.022774,False
5764,BaselineTestE,2019-10-26 00:05:00,baseline recording,0.027043,True


The alert fraction is window-level recall for labeled fault windows and the false-positive fraction for baseline windows. Alert starts per scored hour counts consecutive positives as one alert. The threshold optimizes window F1 on validation as a starting point. Scores can reflect recording conditions, and the same fault recordings contribute to all three splits. For a later Arduino replay, reproduce the within-minute features, feature order and threshold exactly and verify prediction parity on the held-out windows.

## Export and reuse

Run this cell after training. It saves a native LightGBM model and its prediction settings under `models/`. In another notebook, calculate the same one-minute features first. With a feature DataFrame named `features`, use:

```python
import json
from pathlib import Path
import lightgbm as lgb
import numpy as np

settings = json.loads(Path("models/binary.json").read_text(encoding="utf-8"))
classifier = lgb.Booster(model_file="models/binary.txt")
scores = classifier.predict(features[settings["feature_names"]])
class_positions = (scores >= settings["threshold"]).astype(int)
predictions = np.asarray(settings["labels"])[class_positions]
```


In [ ]:
import json
from lightgbm import Booster

export_dir = Path("models")
export_dir.mkdir(exist_ok=True)
model_path = export_dir / "binary.txt"
settings_path = export_dir / "binary.json"
settings = {
    "feature_names": feature_names,
    "input_channels": AIR,
    "window_minutes": LOOKBACK,
    "class_ids": model.classes_.tolist(),
    "labels": ["Baseline", "Anomaly"],
    "threshold": threshold,
}
model.booster_.save_model(str(model_path))
settings_path.write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")

# Verify the exported files reproduce the held-out predictions.
settings = json.loads(settings_path.read_text(encoding="utf-8"))
restored = Booster(model_file=str(model_path))
scores = restored.predict(test[settings["feature_names"]], num_threads=2)
np.testing.assert_allclose(scores, model.booster_.predict(test[feature_names], num_threads=2))
np.testing.assert_array_equal((scores >= threshold).astype(int), test["Alert"])
print(f"Saved {model_path} and {settings_path}. Held-out predictions match.")
